# Three incidences fill a box
### A spatial construction for three pairwise coprime integers

The [two-dimensional construction](02_floor_sum_proof.ipynb) extends to three
stepped solids. For **pairwise coprime** integers $a,b,c>1$,

\[
\boxed{\begin{aligned}
&\sum_{x=1}^{a-1}\left\lfloor\frac{bx}{a}\right\rfloor\left\lfloor\frac{cx}{a}\right\rfloor
+\sum_{y=1}^{b-1}\left\lfloor\frac{ay}{b}\right\rfloor\left\lfloor\frac{cy}{b}\right\rfloor\\
&\quad+\sum_{z=1}^{c-1}\left\lfloor\frac{az}{c}\right\rfloor\left\lfloor\frac{bz}{c}\right\rfloor
=(a-1)(b-1)(c-1).
\end{aligned}}
\]

For $(a,b,c)=(11,7,5)$, the three volumes are $86+80+74=240$.
Each term now counts **rectangular cross-sections**, hence a product of two floors.
Volume means one unit cell per integer triple in the stated domain. Display gaps
between voxels are styling; the renderer's mesh volume is not the counted measure.

Use the **Kaleion** kernel and run all cells. Rotate the figures, isolate each
incidence, scrub the cross-sections, and play the recorded packing and undo.
Installation is unchanged; see [README.md](README.md).

In [ ]:
from pathlib import Path
from math import gcd, prod
from itertools import combinations
import json

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from kaleion import Collection, F, Motion, Workspace, choose, param, vector

pio.renderers.default = "plotly_mimetype+notebook"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "src/kaleion").is_dir())
OUTPUT = ROOT / "build" / "notebooks" / "three-incidences"
OUTPUT.mkdir(parents=True, exist_ok=True)
PARAMETERS = {"a": 11, "b": 7, "c": 5}
assert all(isinstance(v, int) and not isinstance(v, bool) and v > 1
           for v in PARAMETERS.values())
# This mesh viewer is intended for small, fully displayed investigations.
assert prod(v - 1 for v in PARAMETERS.values()) <= 1500
a, b, c = param("a"), param("b"), param("c")

## 1 · Give every point to its greatest normalized coordinate

\[
\begin{aligned}
D&=\{1,\ldots,a-1\}\times\{1,\ldots,b-1\}\times\{1,\ldots,c-1\},\\
X&=\{(x,y,z)\in D:x/a\ge y/b,\ x/a\ge z/c\},\\
Y&=\{(x,y,z)\in D:y/b\ge x/a,\ y/b\ge z/c\},\\
Z&=\{(x,y,z)\in D:z/c\ge x/a,\ z/c\ge y/b\}.
\end{aligned}
\]

All three lenses share the **same occurrence universe**. We keep the non-strict
inequalities so ties remain visible if the coprimality assumption is relaxed.
Membership uses exact integer cross-products. The annotations `u`, `v`, `w`
are mathematical $x,y,z$; floating placement fields `F.x`, `F.y`, `F.z` are
used only by the viewer.

In [ ]:
box = (Collection.grid(a - 1, b - 1, c - 1, values=1)
       .annotate(u=F.i + 1, v=F.j + 1, w=F.k + 1)
       .arrange(F.u, F.v, F.w))
x_rule = (a * F.v <= b * F.u) & (a * F.w <= c * F.u)
y_rule = (b * F.u <= a * F.v) & (b * F.w <= c * F.v)
z_rule = (c * F.u <= a * F.w) & (c * F.v <= b * F.w)
X, Y, Z = box.where(x_rule), box.where(y_rule), box.where(z_rule)

x_sections = X.count(by=F.u)
y_sections = Y.count(by=F.v)
z_sections = Z.count(by=F.w)
ROOTS = {
    "D": box, "X": X, "Y": Y, "Z": Z,
    "X_sections": x_sections, "Y_sections": y_sections, "Z_sections": z_sections,
    "X_volume": x_sections.sum(), "Y_volume": y_sections.sum(), "Z_volume": z_sections.sum(),
    "union": X | Y | Z, "XY": X & Y, "XZ": X & Z, "YZ": Y & Z,
    "XYZ": X & Y & Z, "volume": box.count(),
}
workspace = Workspace(ROOTS, PARAMETERS)
state = workspace.state
assert not state.errors, state.errors

In [ ]:
def inspect_box(state):
    if state.errors:
        raise ValueError(dict(state.errors))
    r = state.results
    av, bv, cv = (state.parameters[k] for k in ("a", "b", "c"))
    extents = (av, bv, cv)
    counts = []
    for axis, name in enumerate("XYZ"):
        others = [extents[k] for k in range(3) if k != axis]
        expected = [(others[0] * t // extents[axis]) * (others[1] * t // extents[axis])
                    for t in range(1, extents[axis])]
        assert r[name + "_sections"].values.tolist() == expected
        total = int(r[name + "_volume"].values[0])
        assert total == r[name].cardinality == sum(expected)
        assert r[name].source.ids == r["D"].ids
        counts.append(total)
    pair_counts = [r[k].cardinality for k in ("XY", "XZ", "YZ")]
    triple = r["XYZ"].cardinality
    volume = int(r["volume"].values[0])
    assert r["union"].mask.all()
    assert r["union"].cardinality == volume == (av - 1) * (bv - 1) * (cv - 1)
    assert sum(counts) - sum(pair_counts) + triple == volume
    pairwise = all(gcd(p, q) == 1 for p, q in combinations(extents, 2))
    if pairwise:
        assert pair_counts == [0, 0, 0] and triple == 0
    return {"parameters": dict(state.parameters), "counts": counts, "volume": volume,
            "pair_intersections": pair_counts, "triple_intersection": triple,
            "pairwise_coprime": pairwise, "disjoint": not any(pair_counts),
            "case_verified": True}

report = inspect_box(state)
for name in "XYZ":
    print(name, "cross-section areas:", state.results[name + "_sections"].values.tolist())
print(report)

## 2 · Inspect the three solids

Rotate by dragging. **X**, **Y**, and **Z** isolate one incidence; **All** restores
the box. **Shared** shows any point belonging to more than one incidence.
Hover a voxel for its exact integer center and membership.

Colors label membership sets, so a shared point is drawn once with its full
membership rather than assigned arbitrarily to a winner. The following collapsed
cell contains only presentation helpers consuming captured results.

In [ ]:
COLORS = {1: "#47bfa7", 2: "#eeb65d", 4: "#9b9ef3",
          3: "#ed83ad", 5: "#ed83ad", 6: "#ed83ad", 7: "#f37968"}
CORNERS = np.array([[0,0,0], [1,0,0], [1,1,0], [0,1,0],
                    [0,0,1], [1,0,1], [1,1,1], [0,1,1]], dtype=float)
TRIANGLES = np.array([[0,2,1], [0,3,2], [4,5,6], [4,6,7],
                      [0,1,5], [0,5,4], [1,2,6], [1,6,5],
                      [2,3,7], [2,7,6], [3,0,4], [3,4,7]])

def membership_name(code):
    return " ∩ ".join(name for bit, name in [(1, "X"), (2, "Y"), (4, "Z")] if code & bit)

def membership_codes(state):
    r = state.results
    return sum(bit * r[name].mask.astype(int) for bit, name in [(1,"X"),(2,"Y"),(4,"Z")])

def mesh_vertices(positions):
    # Nearly unit cubes leave a small visual gap; cardinality measures unit cells.
    return (np.asarray(positions)[:, None, :] + (CORNERS - 0.5) * 0.94).reshape(-1, 3)

def voxel_trace(positions, centers, code):
    vertices = mesh_vertices(positions)
    triangles = (TRIANGLES[None, :, :] + 8 * np.arange(len(positions))[:, None, None]).reshape(-1, 3)
    hover = [f"({int(x)}, {int(y)}, {int(z)}) · {membership_name(code)}" for x,y,z in centers]
    return go.Mesh3d(
        x=vertices[:,0].tolist(), y=vertices[:,1].tolist(), z=vertices[:,2].tolist(),
        i=triangles[:,0].tolist(), j=triangles[:,1].tolist(), k=triangles[:,2].tolist(),
        color=COLORS[code], name=f"{membership_name(code)} · {len(positions)} cells", showlegend=True,
        customdata=np.repeat(hover, 8).tolist(), hovertemplate="%{customdata}<extra></extra>",
        flatshading=True, lighting=dict(ambient=0.75, diffuse=0.65, specular=0.1),
        lightposition=dict(x=100, y=200, z=300),
    )

def box_outline(extents):
    corners = 0.5 + CORNERS * (np.asarray(extents) - 1)
    edges = [(i,j) for i in range(8) for j in range(i+1,8)
             if np.count_nonzero(CORNERS[i] != CORNERS[j]) == 1]
    lines = []
    for i,j in edges:
        lines.extend([corners[i].tolist(), corners[j].tolist(), [None,None,None]])
    return go.Scatter3d(x=[p[0] for p in lines], y=[p[1] for p in lines], z=[p[2] for p in lines],
                        mode="lines", line=dict(color="#9cacbf", width=2),
                        hoverinfo="skip", showlegend=False)

def volume_figure(state, *, positions=None, controls=True):
    r = state.results
    codes = membership_codes(state)
    signatures = sorted(set(codes.tolist()))
    centers = r["D"].positions
    positions = centers if positions is None else positions
    extents = [state.parameters[k] for k in ("a","b","c")]
    title = f"{extents[0]} · {extents[1]} · {extents[2]} — three incidences, {len(centers)} unit cells"
    fig = go.Figure([voxel_trace(positions[codes == k], centers[codes == k], k) for k in signatures])
    fig.add_trace(box_outline(extents))
    axes = {axis + "axis": dict(title=axis, range=[0, extent], autorange=False,
                                 dtick=max(1, extent // 6), backgroundcolor="#101827", gridcolor="#293449")
            for axis, extent in zip("xyz", extents)}
    fig.update_layout(
        template="plotly_dark", paper_bgcolor="#101827", height=650,
        title=dict(text=title, x=0.03, y=0.96, yanchor="top"), margin=dict(l=10,r=10,t=110,b=75),
        scene=dict(**axes, aspectmode="data", camera=dict(eye=dict(x=1.6,y=1.6,z=1.3)),
                   uirevision="three-incidences-camera"), uirevision="three-incidences-camera",
        legend=dict(orientation="h", x=0.02, y=-0.02),
    )
    if controls:
        stages = [("All", lambda k: True), ("X", lambda k: bool(k & 1)),
                  ("Y", lambda k: bool(k & 2)), ("Z", lambda k: bool(k & 4)),
                  ("Shared", lambda k: k not in (1,2,4))]
        fig.update_layout(updatemenus=[dict(
            type="buttons", direction="right", x=0, y=1.02, xanchor="left", yanchor="bottom",
            bgcolor="#293449", showactive=False,
            buttons=[dict(label=name, method="update",
                          args=[{"visible": [rule(k) for k in signatures] + [True]},
                                {"title.text": title + "<br><sup>" + name + "</sup>"}])
                     for name, rule in stages])])
    return fig

In [ ]:
solid_plot = volume_figure(state)
solid_plot.show()

## 3 · A cross-section is a product of two floor quotients

At fixed $x$, the $X$ section is the rectangle

\[
1\le y\le\left\lfloor bx/a\right\rfloor,\qquad
1\le z\le\left\lfloor cx/a\right\rfloor.
\]

Its area is their product. The other incidences fill the rest of this section.
The scrubber below visits **exact integer slices**, with their counted areas.
Change `SLICE_AXIS` to `"y"` or `"z"` to inspect the other products.
An empty section contributes zero and remains in the derived arrangement.

In [ ]:
def slice_figure(state, axis="x"):
    axis_index = "xyz".index(axis)
    extents = [state.parameters[k] for k in ("a","b","c")]
    shape = tuple(v - 1 for v in extents)
    remaining = [j for j in range(3) if j != axis_index]
    codes = membership_codes(state).reshape(shape)
    captured = state.results[axis.upper() + "_sections"].values
    # This view is used for disjoint incidences, where the only codes are 1, 2, 4.
    assert set(np.unique(codes)).issubset({1, 2, 4})
    color_scale = [[0,COLORS[1]],[0.25,COLORS[1]],
                   [0.25,COLORS[2]],[0.75,COLORS[2]],[0.75,COLORS[4]],[1,COLORS[4]]]
    def slice_data(t):
        raw = np.take(codes, t, axis=axis_index).T
        return np.where(raw == 1, 0, np.where(raw == 2, 1, 2)).tolist()
    def title(t):
        return f"{axis} = {t+1} · {axis.upper()} cross-section = {int(captured[t])} unit cells"
    start = (shape[axis_index] - 1) // 2
    frames = [go.Frame(name=str(t), data=[go.Heatmap(z=slice_data(t))], traces=[0],
                       layout=dict(title=dict(text=title(t)))) for t in range(shape[axis_index])]
    fig = go.Figure(data=[go.Heatmap(
        x=list(range(1, extents[remaining[0]])), y=list(range(1, extents[remaining[1]])),
        z=slice_data(start), zmin=0, zmax=2, colorscale=color_scale, xgap=2, ygap=2, showscale=False,
        hovertemplate=f"{'xyz'[remaining[0]]} = %{{x}}, {'xyz'[remaining[1]]} = %{{y}}<extra></extra>",
    )], frames=frames)
    instant = dict(mode="immediate", frame=dict(duration=0,redraw=True), transition=dict(duration=0))
    fig.update_layout(
        template="plotly_dark", paper_bgcolor="#101827", plot_bgcolor="#101827", height=460,
        title=dict(text=title(start),x=0.04), margin=dict(l=55,r=35,t=65,b=100),
        xaxis=dict(title="xyz"[remaining[0]], range=[0.5,extents[remaining[0]]-0.5],dtick=1,constrain="domain"),
        yaxis=dict(title="xyz"[remaining[1]], range=[0.5,extents[remaining[1]]-0.5],dtick=1,
                   scaleanchor="x",scaleratio=1,constrain="domain"),
        sliders=[dict(active=start, currentvalue=dict(prefix=axis+" = "),
                      steps=[dict(label=str(t+1),method="animate",args=[[str(t)],instant])
                             for t in range(shape[axis_index])])],
    )
    return fig

In [ ]:
SLICE_AXIS = "x"
section_plot = None
if report["disjoint"]:
    section_plot = slice_figure(state, SLICE_AXIS)
    section_plot.show()
else:
    display(Markdown("These parameters have shared points. Inspect **Shared** in the solid view."))

## 4 · Assemble the box, then undo along the recorded path

Translate $X$ along $x$, $Y$ along $y$, and $Z$ along $z$ to separate the three
pieces. Packing restores their original positions; undo traverses the recorded
path in reverse. Colors follow occurrence identities, and every mathematical
membership remains attached to its captured integer point throughout the motion.

The motion is an illustration of the partition. Intersections of moving display
meshes are not new mathematical intersections. For parameters with shared points,
the disjoint packing is omitted and the overlaps are shown explicitly instead.

In [ ]:
packing_workspace = None
motion_frames = []
if report["disjoint"]:
    separated = box.move(vector(choose(x_rule, a, 0),
                                choose(y_rule, b, 0), choose(z_rule, c, 0)))
    packing_workspace = Workspace({"tiles": separated}, PARAMETERS)
    packed = packing_workspace.set("tiles", box, motion=Motion())
    undone = packing_workspace.undo()
    times = np.linspace(0, 1, 31)
    motion_frames = [step.frame("tiles", float(t)) for step in (packed, undone) for t in times]
    motion_labels = [f"{verb} · {t:.0%}" for verb in ("Fill", "Undo") for t in times]
    np.testing.assert_allclose(packed.frame("tiles", 1).positions,
                               state.results["D"].positions, rtol=0, atol=1e-12)
    np.testing.assert_allclose(undone.frame("tiles", 1).positions,
                               packed.frame("tiles", 0).positions, rtol=0, atol=1e-12)

In [ ]:
def motion_figure(state, samples, labels):
    ids = state.results["D"].ids
    codes = membership_codes(state)
    signatures = sorted(set(codes.tolist()))
    positions = []
    for sample in samples:
        lookup = {oid: i for i, oid in enumerate(sample.after_ids)}
        assert set(lookup) == set(ids)
        positions.append(sample.positions[[lookup[oid] for oid in ids]])
    fig = volume_figure(state, positions=positions[0], controls=False)
    title = " + ".join(str(int(state.results[n+"_volume"].values[0])) for n in "XYZ")
    title += f" = {len(ids)} unit cells"
    frames = []
    for i, points in enumerate(positions):
        data = []
        for k in signatures:
            vertices = mesh_vertices(points[codes == k])
            data.append(go.Mesh3d(x=vertices[:,0].tolist(), y=vertices[:,1].tolist(), z=vertices[:,2].tolist()))
        frames.append(go.Frame(name=str(i), data=data, traces=list(range(len(signatures))),
                               layout=dict(title=dict(text=title+"<br><sup>"+labels[i]+"</sup>"))))
    fig.frames = frames
    all_points = np.concatenate(positions)
    scene = {axis+"axis": dict(range=[float(all_points[:,j].min())-0.6,
                                       float(all_points[:,j].max())+0.6],
                              dtick=max(1, int(np.ceil(np.ptp(all_points[:,j]) / 7))))
             for j, axis in enumerate("xyz")}
    instant = dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))
    play = dict(mode="immediate", frame=dict(duration=60, redraw=True), transition=dict(duration=0), fromcurrent=True)
    fig.update_layout(
        scene=scene, title=dict(text=title+"<br><sup>"+labels[0]+"</sup>"),
        height=700, margin=dict(l=10,r=10,t=85,b=160),
        legend=dict(y=-0.02),
        updatemenus=[dict(type="buttons",direction="right",x=0,y=-0.1,xanchor="left",yanchor="top",
                          bgcolor="#293449",showactive=False,
                          buttons=[dict(label="Play",method="animate",args=[None,play]),
                                   dict(label="Pause",method="animate",args=[[None],instant]),
                                   dict(label="Restart",method="animate",args=[["0"],instant])])],
        sliders=[dict(y=-0.24,currentvalue=dict(visible=False),
                      steps=[dict(label=label,method="animate",args=[[str(i)],instant])
                             for i,label in enumerate(labels)])],
    )
    return fig

packing_plot = None
if motion_frames:
    packing_plot = motion_figure(state, motion_frames, motion_labels)
    packing_plot.show()

## 5 · The general proof

**Coverage.** Every triple $(x/a,y/b,z/c)$ has a largest coordinate, so
$X\cup Y\cup Z=D$.

**No ties.** For example, $x/a=y/b$ would give $bx=ay$.
Since $\gcd(a,b)=1$, this forces $a\mid x$, impossible for $1\le x<a$.
The same reasoning applies to the other two pairs. Thus the largest coordinate
is unique, and $X,Y,Z$ are disjoint.

**Count cross-sections.** For fixed $x$, the inequalities defining $X$ bound $y$
and $z$ independently. All valid combinations occur in the Cartesian box, giving
$\lfloor bx/a\rfloor\lfloor cx/a\rfloor$ points. Summing over $x$ gives $|X|$;
the other two sums give $|Y|$ and $|Z|$.

Consequently the sum of the three floor-product sums equals
$|D|=(a-1)(b-1)(c-1)$. This deduction proves the general statement.
The finite assertions and recorded motion illustrate selected cases; they are
not an automated proof certificate.

## 6 · Three numbers with gcd 1 need not be pairwise coprime

For $(a,b,c)=(6,4,5)$, $\gcd(6,4,5)=1$ but $\gcd(6,4)=2$.
The two points $(3,2,1)$ and $(3,2,2)$ have $x/6=y/4=1/2\ge z/5$,
so both belong to $X\cap Y$. The three incidence counts are $23,19,20$,
while the box has $60$ cells: $23+19+20-2=60$.

Click **Shared** below to isolate these two cells. Equality of two normalized
coordinates matters for overlap only when they are jointly maximal.
For arbitrary parameters, the exact accounting is inclusion–exclusion:

\[
|X|+|Y|+|Z|-|X\cap Y|-|X\cap Z|-|Y\cap Z|+|X\cap Y\cap Z|=|D|.
\]

For example, $(4,6,8)$ also has a triple intersection: the counts total $118$,
pairwise intersections have sizes $4,8,2$, and the triple intersection has size
$1$, giving $118-4-8-2+1=105$. Ties are retained as intersections.

In [ ]:
noncoprime_workspace = Workspace(ROOTS, {"a": 6, "b": 4, "c": 5})
noncoprime_report = inspect_box(noncoprime_workspace.state)
overlap = noncoprime_workspace.state.results["XY"]
shared = list(zip(*(overlap.source.fields[k][overlap.mask].tolist() for k in ("u","v","w"))))
assert shared == [(3, 2, 1), (3, 2, 2)]
print("Shared coordinates:", shared)
print(noncoprime_report)
overlap_plot = volume_figure(noncoprime_workspace.state)
overlap_plot.show()

triple_workspace = Workspace(ROOTS, {"a": 4, "b": 6, "c": 8})
triple_report = inspect_box(triple_workspace.state)
assert triple_report["pair_intersections"] == [4, 8, 2]
assert triple_report["triple_intersection"] == 1
print("A case with a triple intersection:", triple_report)

## 7 · Retain the investigation

The definitions, incidence intersections, grouped areas, volumes, and their
provenance are workspace roots. Captures record the finite statements checked
here. Packing history is saved separately so its undo path can be replayed.
Standalone HTML exports embed Plotly.js and need no CDN. The full converted
notebook may use external assets to typeset mathematics.

In [ ]:
for name, investigation, case in [("box",workspace,report),
                                   ("overlap",noncoprime_workspace,noncoprime_report),
                                   ("triple",triple_workspace,triple_report)]:
    investigation.capture("Finite box-incidence verification: " + json.dumps(case) +
                          ". The general argument is written in the notebook.")
    (OUTPUT / f"{name}-workspace.json").write_text(investigation.to_json(), encoding="utf-8")
if packing_workspace is not None:
    (OUTPUT / "packing-workspace.json").write_text(packing_workspace.to_json(), encoding="utf-8")
for name, fig in [("solids",solid_plot), ("sections",section_plot),
                  ("packing",packing_plot), ("overlap",overlap_plot)]:
    if fig is not None:
        fig.write_html(OUTPUT / f"{name}.html", include_plotlyjs=True, auto_play=False)
(OUTPUT / "cases.json").write_text(json.dumps([report,noncoprime_report,triple_report], indent=2), encoding="utf-8")
snapshot = state.results["D"]
preview = {"parameters": dict(state.parameters), "counts": report["counts"],
           "cells": [[int(snapshot.fields[k][i]) for k in ("u","v","w")] + [int(signature)]
                     for i,signature in enumerate(membership_codes(state))]}
(OUTPUT / "preview.json").write_text(json.dumps(preview), encoding="utf-8")
print("Saved interactive figures, exact case reports, and workspace histories to", OUTPUT)

The same counting idea extends mathematically to any number of pairwise coprime
side parameters: assign each interior lattice point to its unique greatest
normalized coordinate. In $n$ dimensions, each summand becomes a product of
$n-1$ floor quotients. The change from area to volume was the first visible step
of that general construction.